# Kata kunci TF-IDF
        **text › keywords_summary** · notebook `tfidf_keywords`

        **Kapan dipakai:** Kata kunci per dokumen berdasarkan bobot TF-IDF terhadap korpus.

        **Jebakan umum:** Butuh korpus pembanding; frasa multikata tidak tertangkap kecuali bigram.

> Evaluasi di sini **kualitatif** (dibaca manusia). Metrik otomatis (ROUGE) butuh ringkasan referensi.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/text/classification_en/news4_train.csv"
TEST_PATH  = ROOT / "data_latihan/text/classification_en/news4_test.csv"
TEXT_COL   = "text"
LABEL_COL  = "label"            # hanya untuk EVALUASI; algoritma tidak melihatnya
LANG       = "en"
USE_STOPWORDS = True
SEED = 42
MODEL_KEY  = "tfidf_keywords"
OUT = out_dir("text", "keywords_summary", MODEL_KEY)
set_seed(SEED)
from superdb.text import ID_STOPWORDS
STOP = (ID_STOPWORDS if LANG == "id" else "english") if USE_STOPWORDS else None
DOC_IDX = [0, 5, 11]      # dokumen yang dicoba
TOP_K, N_SENT = 8, 3

## 2. Baca dokumen

In [ ]:
from superdb.text_extra import split_sentences, textrank_keywords, textrank_summary
df = pd.read_csv(TRAIN_PATH); docs = df[TEXT_COL].astype(str).tolist()
for i in DOC_IDX:
    print(f"[{i}] ({df[LABEL_COL][i]}) {len(docs[i].split())} kata, {len(split_sentences(docs[i]))} kalimat: {docs[i][:150]}...")

## 3. Jalankan metode

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.5, stop_words=STOP, sublinear_tf=True).fit(docs)
terms = np.asarray(vec.get_feature_names_out())
result = {}
for i in DOC_IDX:
    w = vec.transform([docs[i]]).toarray().ravel(); result[i] = list(terms[np.argsort(-w)[:TOP_K]])
for i, k in result.items(): print(f"[{i}] kata kunci:", ", ".join(k))

## 4. Simpan

In [ ]:
save_json({str(k): v for k, v in result.items()}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Bandingkan metode berdampingan pada dokumen yang sama; **lead baseline wajib dikalahkan** sebelum mengklaim ringkasan lebih baik.
- Ada ringkasan referensi di data lomba? Hitung ROUGE (paket `rouge-score`; pasang lewat `tools/pipsafe.py`).
- Ringkasan abstraktif (menulis ulang) butuh model seq2seq (folder heavy).